In [ ]:
from __future__ import annotations

import re
import json as _json
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
import wandb

# ── Font / style (matches paper_plots_with_ami.ipynb, sized up for readability
# in this large per-model 7x4 grid) ──────────────────────────────────────────
LABEL_FONTSIZE           = 26
TITLE_FONTSIZE           = 28
TICK_FONTSIZE            = 24
CATEGORY_TICK_FONTSIZE   = 20
ROWLABEL_FONTSIZE        = 30
SUPTITLE_FONTSIZE        = 34
CBAR_TICK_FONTSIZE       = 26
CBAR_LABEL_FONTSIZE      = 30
LEGEND_FONTSIZE          = 20

HEATMAP_CMAP = 'RdBu_r'
VMIN, VMAX   = -1.0, 1.0
Y_MAX     = 0.5
Y_TICKS   = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
N_X_TICKS = 5

MAX_CATEGORIES = 60

# klsm (KL, softmax-normalized) is still computed and logged by main.py
# alongside kl (KL, L1-normalized) for comparison, but this notebook only
# plots kl (L1-normalized) -- displayed simply as "KL" since it's the only
# KL variant shown here.
METRICS = ["l1", "kl", "saecos", "dot"]
METRIC_LABELS = {"l1": "L1", "kl": "KL", "saecos": "SAE cos", "dot": "SAE dot"}
METRIC_COLORS = {"l1": "#ff7f0e", "kl": "#2ca02c", "saecos": "#d62728", "dot": "#9467bd"}
EMB_TYP_COLOR = "#333333"   # human vs emb-metric, within the combined embedding-side row
EMB_EMB_COLOR = "#1f9e89"   # static-emb vs emb-metric, within the combined embedding-side row

# Row 0 is AMI (layer-indexed line plot), rows 1-4 are layer-indexed heatmaps,
# row 5 is the layer x layer heatmap, row 6 is a category-indexed line plot
# combining both single-layer "emb" comparisons (human & static-emb vs emb-metric).
# Wrapped in \mathbf{...} because matplotlib mathtext does not honor the
# Text object's fontweight='bold' for embedded math -- only plain text (row 0)
# bolds via fontweight alone, so the math rows need an explicit \mathbf.
ROW_TITLES = [
    r'AMI vs layer',
    r'$\mathbf{\rho_{\mathrm{S}}(s_{\mathrm{hum}},\, s_{\mathrm{M}}^\ell)}$',
    r'$\mathbf{\rho_{\mathrm{S}}(s_{\mathrm{emb}},\, s_{\mathrm{M}}^\ell)}$',
    r'$\mathbf{\rho_{\mathrm{S}}(s_{\mathrm{cos}}^\ell,\, s_{\mathrm{M}}^\ell)}$',
    r'$\mathbf{\rho_{\mathrm{S}}(s_{\mathrm{M}}^\ell,\, s_{\mathrm{Jac}}^\ell)}$',
    r'$\mathbf{\rho_{\mathrm{S}}(s_{\mathrm{M}}^{\ell_1},\, s_{\mathrm{M}}^{\ell_2})}$',
    r'$\mathbf{\rho_{\mathrm{S}}(\cdot,\, s_{\mathrm{M,emb}})}$',
]

MODEL_ORDER = [
    'gemma-3-270m', 'gemma-3-1b-pt', 'gemma-3-4b-pt',
    'gemma-3-12b-pt', 'gemma-3-27b-pt',
    'gpt2-small', 'meta-llama/Llama-3.1-8B', 'mistralai/Mistral-7B-v0.1',
]

PRETTY_MODEL = {
    'gemma-3-270m':                'Gemma 3 270M',
    'gemma-3-1b-pt':               'Gemma 3 1B',
    'gemma-3-4b-pt':               'Gemma 3 4B',
    'gemma-3-12b-pt':              'Gemma 3 12B',
    'gemma-3-27b-pt':              'Gemma 3 27B',
    'gpt2-small':                  'GPT-2 Small',
    'meta-llama/Llama-3.1-8B':     'Llama 3.1 8B',
    'mistralai/Mistral-7B-v0.1':   'Mistral 7B',
}

import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
WANDB_PROJECT = 'experiment_1_typicality'  # wandb.project in configs/experiment_1_typicality/base.yaml
WANDB_CACHE   = Path('.wandb_cache_additional_metrics')
WANDB_CACHE.mkdir(parents=True, exist_ok=True)

OUTPUT_DIR = Path('paper_plots_additional_metrics_out')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Plotting primitives

Reused from `paper_plots_with_ami.ipynb` (generic heatmap/layer-layer/colorbar helpers, not Jaccard-specific).

In [ ]:
def _cmap_with_nan():
    cmap = plt.get_cmap(HEATMAP_CMAP).copy()
    cmap.set_bad(color='white')
    return cmap


def sparse_xticks(vals: List[int], n: int = 6) -> List[int]:
    vals = sorted(vals)
    if len(vals) <= n:
        return vals
    idxs = [round(i * (len(vals) - 1) / (n - 1)) for i in range(n)]
    seen, out = set(), []
    for idx in idxs:
        v = vals[idx]
        if v not in seen:
            out.append(v); seen.add(v)
    return out


def get_layer_cols(df: pd.DataFrame) -> List[str]:
    cols = [c for c in df.columns if re.fullmatch(r'layer_\d+', str(c))]
    return sorted(cols, key=lambda c: int(c.split('_')[1]))


def normalize_ll(df: pd.DataFrame) -> pd.DataFrame:
    if 'index' in df.columns:
        df = df.set_index('index')
    df.index   = pd.Index([str(c) for c in df.index])
    df.columns = [str(c) for c in df.columns]
    common = sorted([c for c in df.columns if c in set(df.index)],
                    key=lambda x: int(x) if x.isdigit() else x)
    if common:
        df = df.loc[common, common]
    return df.astype(float)


def draw_blank(ax: Axes, note: Optional[str] = None) -> None:
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(True); sp.set_alpha(0.25)
    ax.set_facecolor('#f8f8f8')
    if note:
        ax.text(0.5, 0.5, note, transform=ax.transAxes, ha='center', va='center',
                fontsize=LABEL_FONTSIZE - 6, style='italic', color='#888888',
                multialignment='center')


def plot_rect_on_ax(
    ax: Axes, mat: np.ndarray,
    row_labels: List[str], col_labels: List[str],
    xlabel: str = '', show_xticks: bool = True, show_xlabel: bool = False,
    show_yticks: bool = True, max_xticks: int = 6,
) -> None:
    """show_xticks controls the numeric layer tick labels (shown on every
    layerwise row); show_xlabel controls only the 'Layer' axis caption text
    (shown once, on the bottom-most layerwise row, to avoid repeating it)."""
    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    ax.imshow(mat, aspect='auto', cmap=_cmap_with_nan(), norm=norm, interpolation='nearest')

    col_vals = [int(x) for x in col_labels]
    ticks    = sparse_xticks(col_vals, n=max_xticks)
    tidx     = [col_vals.index(x) for x in ticks]
    ax.set_xticks(tidx)
    ax.set_xticklabels([str(x) for x in ticks] if show_xticks else [''] * len(tidx),
                       fontsize=TICK_FONTSIZE - 4)

    if show_yticks:
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=CATEGORY_TICK_FONTSIZE - 4)
    else:
        ax.set_yticks([])

    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE - 4, pad=3)
    ax.set_xlabel(xlabel if show_xlabel else '', fontsize=LABEL_FONTSIZE - 4)
    ax.set_xlim(-0.5, mat.shape[1] - 0.5)
    ax.set_ylim(mat.shape[0] - 0.5, -0.5)


def plot_ll_on_ax(
    ax: Axes, df_ll: pd.DataFrame,
    show_xticks: bool = True, show_xlabel: bool = False,
    show_yticks: bool = True, max_ticks: int = 6,
) -> None:
    mat    = df_ll.to_numpy(dtype=float)
    labels = [str(c) for c in df_ll.columns]
    mat_p  = mat.copy()
    mat_p[np.triu(np.ones_like(mat, dtype=bool), k=1)] = np.nan

    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    # aspect='auto' (not 'equal') so this panel fills the same column width as
    # the rectangular category-by-layer heatmaps in the rows above it.
    ax.imshow(mat_p, aspect='auto', cmap=_cmap_with_nan(), norm=norm, interpolation='nearest')

    tick_vals = [int(x) for x in labels]
    ticks     = sparse_xticks(tick_vals, n=max_ticks)
    tidx      = [tick_vals.index(x) for x in ticks]
    tlabels   = [str(x) for x in ticks]

    ax.set_xticks(tidx)
    ax.set_xticklabels(tlabels if show_xticks else [''] * len(tidx), fontsize=TICK_FONTSIZE - 4)
    if show_yticks:
        ax.set_yticks(tidx)
        ax.set_yticklabels(tlabels, fontsize=TICK_FONTSIZE - 4)
    else:
        ax.set_yticks([])
    ax.tick_params(axis='both', labelsize=TICK_FONTSIZE - 4, pad=3)
    ax.set_xlabel(r'Layer $\ell_1$' if show_xlabel else '', fontsize=LABEL_FONTSIZE - 4)
    ax.set_ylabel(r'Layer $\ell_2$' if show_yticks else '', fontsize=LABEL_FONTSIZE - 4)
    ax.set_xlim(-0.5, mat.shape[1] - 0.5)
    ax.set_ylim(mat.shape[0] - 0.5, -0.5)


def add_shared_colorbar(fig, bottom: float, top: float, x: float = 0.965, width: float = 0.010) -> None:
    norm = mcolors.TwoSlopeNorm(vmin=VMIN, vcenter=0.0, vmax=VMAX)
    sm   = ScalarMappable(norm=norm, cmap=plt.get_cmap(HEATMAP_CMAP))
    sm.set_array([])
    cax  = fig.add_axes([x, bottom, width, top - bottom])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label(r'Spearman $\rho$', fontsize=CBAR_LABEL_FONTSIZE - 4, labelpad=10)
    cbar.ax.tick_params(labelsize=CBAR_TICK_FONTSIZE - 4)


def _parse_layer(rep_str: str) -> Optional[int]:
    m = re.search(r'_(\d+)$', rep_str)
    return int(m.group(1)) if m else None

## W&B discovery helpers

Adapted from `paper_plots_with_ami.ipynb` (generic model/SAE-label parsing and table download logic, unchanged).

In [ ]:
_GEMMA_MODEL_MAP = {
    '270m': 'gemma-3-270m',
    '1b':   'gemma-3-1b-pt',
    '4b':   'gemma-3-4b-pt',
    '12b':  'gemma-3-12b-pt',
    '27b':  'gemma-3-27b-pt',
}


def _wcfg(cfg, *keys, default=None):
    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def _sae_label_from_id_template(s: str) -> Optional[str]:
    s = s.lower()
    m = re.search(r'width[_-](\d+k)[_-]l\d+[_-](big|small|medium|canonical)', s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.search(r'(\d+k).+?(big|small|medium|canonical)', s)
    if m:
        return f"{m.group(1)} {m.group(2)}"
    m = re.search(r'(\d+k)', s)
    if m:
        return m.group(1)
    return None


def _normalize_sae_model_name(model_part: str) -> Optional[str]:
    if model_part in MODEL_ORDER:
        return model_part
    m = re.match(r'gemma-3-?(270m|1b|4b|12b|27b)$', model_part, re.I)
    if m:
        return _GEMMA_MODEL_MAP.get(m.group(1).lower())
    if re.search(r'llama', model_part, re.I) and re.search(r'8b', model_part, re.I):
        return 'meta-llama/Llama-3.1-8B'
    if re.search(r'mistral', model_part, re.I) and re.search(r'7b', model_part, re.I):
        return 'mistralai/Mistral-7B-v0.1'
    if re.search(r'gpt2', model_part, re.I):
        return 'gpt2-small'
    return None


def _get_wandb_model_and_label(run) -> Tuple[Optional[str], Optional[str]]:
    cfg = run.config or {}
    sae = cfg.get('sae')
    if isinstance(sae, dict):
        model_name  = _wcfg(sae, 'model', 'name')
        id_template = sae.get('id_template', '')
        label = _sae_label_from_id_template(id_template or '')
        if model_name and model_name in MODEL_ORDER:
            return model_name, label
    sae_str = str(sae) if sae is not None else ''
    if '/' in sae_str:
        model_part, label_part = sae_str.split('/', 1)
        model_name = _normalize_sae_model_name(model_part.strip())
        label      = label_part.strip().replace('_', ' ')
        if model_name:
            return model_name, label
    return None, None

# Llama Scope 32x has 2**17 = 131,072 latents and is called "128k" by its release, like GPT-2 v5-128k:
# label both "128k". Older runs/configs call the Llama SAE "131k".
def _width_128k(label):
    return "128k" if label == "131k" else label


# 131k -> 128k (see _width_128k); wrapped here, before the W&B runs are grouped by label below
__get_wandb_model_and_label_raw = _get_wandb_model_and_label


def _get_wandb_model_and_label(*args, **kwargs):
    model_name, label = __get_wandb_model_and_label_raw(*args, **kwargs)
    return model_name, _width_128k(label)



def _index_wandb_tables(run, logical_names: List[str]) -> Dict[str, object]:
    """Match every logical_name (the wandb key with the 'tables/' prefix
    stripped, e.g. 'df_cat_typ_l1_by_layer' for a table logged as
    'tables/df_cat_typ_l1_by_layer') against a single listing of this run's
    files -- confirmed empirically against
    media/table/tables/{logical_name}_{step}_{hash}.table.json.

    Doing the listing once per run (instead of once per logical_name, as a
    prior version did) matters a lot: run.files() is a paginated W&B API
    call, and with ~34 logical names and dozens of runs, re-listing per name
    turned the fetch step into 1000+ API calls -- slow enough that the whole
    notebook looked like it produced no output at all.
    """
    table_files = [f for f in run.files() if f.name.endswith('.table.json')]
    index: Dict[str, object] = {}
    for name in logical_names:
        pat = re.compile(rf'^{re.escape(name)}_\d+_')
        hits = sorted(
            (f for f in table_files if pat.match(Path(f.name).name)),
            key=lambda x: x.name,
        )
        if hits:
            index[name] = hits[0]
    return index


def _parse_table_json(path: Path) -> pd.DataFrame:
    with path.open('r', encoding='utf-8') as fp:
        obj = _json.load(fp)
    return pd.DataFrame(obj['data'], columns=obj['columns'])


def _download_wandb_table(file_obj, cache_dir: Path) -> Path:
    cache_dir.mkdir(parents=True, exist_ok=True)
    return Path(file_obj.download(root=str(cache_dir), replace=False, exist_ok=True).name)

## Discover runs and download tables

Wandb-only (no local CSV merging) — reads directly from
`WANDB_ENTITY/WANDB_PROJECT`. Only `finished`
runs are considered; the most recently created run per (model, SAE label) wins.

Missing tables (e.g. the metric-vs-Jaccard comparison and per-metric AMI
before a rerun with the updated `main.py` lands) are simply absent from the
per-run dict and rendered as blank placeholders later — nothing errors.

In [ ]:
def per_metric_table_keys(m: str) -> Dict[str, str]:
    return {
        'typ_by_layer': f'df_cat_typ_{m}_by_layer',
        'emb_by_layer': f'df_cat_emb_{m}_by_layer',
        'ctx_by_layer': f'df_cat_ctx_{m}_by_layer',
        'jac_by_layer': f'df_cat_{m}_jac_by_layer',
        'll':           f'df_ll_{m}',
        'typ_emb_agg':  f'df_cat_typ_emb_{m}',
        'emb_emb_agg':  f'df_cat_emb_emb_{m}',
    }

# Jaccard's own tables predate the per-metric naming convention above (it has
# no "vs Jaccard" row of its own, and no "_jac_" infix -- these are the
# original tables main.py has always logged for the Jaccard baseline).
JACCARD_TABLE_KEYS = {
    'typ_by_layer': 'df_cat_typ_jac_by_layer',
    'emb_by_layer': 'df_cat_emb_jac_by_layer',
    'ctx_by_layer': 'df_cat_ctx_jac_by_layer',
    'typ_emb_agg':  'df_cat_typ_emb_jac',
    'emb_emb_agg':  'df_cat_emb_cosim_emb_jac',
}

ALL_TABLE_KEYS = sorted(
    {v for m in METRICS for v in per_metric_table_keys(m).values()}
    | set(JACCARD_TABLE_KEYS.values())
    | {'df_clustering_ami'}
)
print(f'{len(ALL_TABLE_KEYS)} table keys per run:')
for k in ALL_TABLE_KEYS:
    print(' ', k)

print(f'\nFetching runs from {WANDB_ENTITY}/{WANDB_PROJECT} ...')
_wapi  = wandb.Api(timeout=180)
_wruns = list(_wapi.runs(f'{WANDB_ENTITY}/{WANDB_PROJECT}', per_page=200))
print(f'Found {len(_wruns)} runs total')

_wrun_groups: Dict[Tuple[str, str], list] = defaultdict(list)
for _run in _wruns:
    if _run.state != 'finished':
        continue
    _model, _label = _get_wandb_model_and_label(_run)
    if _model is None or _label is None:
        continue
    _wrun_groups[(_model, _label)].append(_run)

print(f'Groups with finished runs: {sorted(_wrun_groups.keys())}')

model_data: Dict[Tuple[str, str], Dict[str, pd.DataFrame]] = {}
for (_model, _label), _group in sorted(_wrun_groups.items()):
    _best = sorted(_group, key=lambda r: str(getattr(r, 'created_at', '')))[-1]
    _safe_mdl  = _model.replace('/', '__')
    _safe_sae  = _label.replace('/', '__').replace(' ', '_')
    _cache_dir = WANDB_CACHE / _safe_mdl / _safe_sae

    _file_index = _index_wandb_tables(_best, ALL_TABLE_KEYS)
    _tables: Dict[str, pd.DataFrame] = {}
    for _key in ALL_TABLE_KEYS:
        _fobj = _file_index.get(_key)
        if _fobj is None:
            continue
        try:
            _path = _download_wandb_table(_fobj, _cache_dir)
            _tables[_key] = _parse_table_json(_path)
        except Exception as _e:
            print(f'  [wandb] {_key} error for {_model}/{_label}: {_e}')

    model_data[(_model, _label)] = _tables
    print(f'  loaded {_model} / {_label}: {len(_tables)}/{len(ALL_TABLE_KEYS)} tables present')

## Category ordering

Shared across all figures — mean rho (typicality vs L1, by layer) per category, capped at `MAX_CATEGORIES`.

In [ ]:
_rho_acc: Dict[str, List[float]] = defaultdict(list)
for _tables in model_data.values():
    _df = _tables.get(per_metric_table_keys('l1')['typ_by_layer'])
    if _df is None:
        continue
    _lc = get_layer_cols(_df)
    for _, _row in _df.iterrows():
        _rho_acc[str(_row['category'])].append(float(np.nanmean([_row[c] for c in _lc])))

ordered_cats = sorted(_rho_acc.keys())[:MAX_CATEGORIES]
print(f'Categories: {len(ordered_cats)}, first 10: {ordered_cats[:10]}')

## Per-row drawing functions

One function per row *type* (AMI line, by-layer heatmap, layer×layer heatmap, category line),
reused across all 4 metric columns.

In [ ]:
N_ROWS, N_COLS = 7, len(METRICS)


def _plot_ami_row(ax: Axes, df_ami: Optional[pd.DataFrame], m: str, show_xlabel: bool, show_yticks: bool) -> None:
    ax.set_ylim(0.0, Y_MAX)
    ax.set_yticks(Y_TICKS)
    ax.set_yticklabels([f'{v:.1f}' for v in Y_TICKS] if show_yticks else [],
                        fontsize=TICK_FONTSIZE - 6)
    ax.grid(axis='y', alpha=0.30, linewidth=0.8)
    ax.tick_params(labelsize=TICK_FONTSIZE - 6)
    if df_ami is None or 'representation' not in df_ami.columns:
        draw_blank(ax, 'no AMI data\n(rerun needed)')
        return

    # "Dense residual" = k-medoids AMI (vs. true category labels) clustering
    # words by cosine similarity of the dense, prompted residual-stream vector
    # at that layer. "{metric} (SAE)" = same clustering/scoring but using this
    # column's metric on the SAE latent vectors at that layer instead. Same
    # x-axis (layer), same AMI y-axis — directly comparable curves. Styling
    # matches paper_plots_with_ami.ipynb's AMI-overview cell: a thick black
    # marker-less reference line vs. a thinner colored+marker line.
    has_line = False
    xmax = 0
    for pattern, color, label, marker, linewidth in (
        (r'^contextual_layer_\d+$', 'black', 'Dense residual', None, 2.2),
        (rf'^{re.escape(m)}_layer_\d+$', METRIC_COLORS[m], f'{METRIC_LABELS[m]} (SAE)', 'o', 1.4),
    ):
        rows = df_ami[df_ami['representation'].str.match(pattern)]
        pts = sorted(
            (( _parse_layer(r), a) for r, a in zip(rows['representation'], rows['ami'])
             if _parse_layer(r) is not None)
        )
        if pts:
            xs = [p for p, _ in pts]
            ax.plot(xs, [a for _, a in pts], marker=marker, markersize=3.5,
                    linewidth=linewidth, color=color, label=label)
            has_line = True
            xmax = max(xmax, max(xs))

    if not has_line:
        draw_blank(ax, 'no AMI data\n(rerun needed)')
        return

    ax.legend(fontsize=LEGEND_FONTSIZE - 4, loc='lower right', frameon=False,
              handlelength=1.0, borderpad=0.2, labelspacing=0.2)
    ticks = np.linspace(0, xmax, N_X_TICKS, dtype=int).tolist()
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(t) for t in ticks], fontsize=TICK_FONTSIZE - 6)
    if show_xlabel:
        ax.set_xlabel(r'Layer $\ell$', fontsize=LABEL_FONTSIZE - 6)


def _plot_heatmap_row(ax: Axes, df: Optional[pd.DataFrame], show_xlabel: bool, show_yticks: bool) -> None:
    """show_xlabel here only gates the 'Layer' axis-caption text; the numeric
    layer tick labels themselves are always shown (show_xticks=True)."""
    if df is None:
        draw_blank(ax, 'no data\n(rerun needed)')
        return
    lc = get_layer_cols(df)
    if not lc:
        draw_blank(ax, 'no data')
        return
    ll  = [str(int(c.split('_')[1])) for c in lc]
    d   = df.set_index('category').reindex(ordered_cats)
    mat = d[lc].to_numpy(dtype=float)
    plot_rect_on_ax(ax, mat, ordered_cats, ll, xlabel=r'Layer $\ell$',
                     show_xticks=True, show_xlabel=show_xlabel, show_yticks=show_yticks)


def _plot_ll_row(ax: Axes, df_ll_raw: Optional[pd.DataFrame], show_xlabel: bool, show_yticks: bool) -> None:
    if df_ll_raw is None:
        draw_blank(ax, 'no data')
        return
    plot_ll_on_ax(ax, normalize_ll(df_ll_raw), show_xticks=True, show_xlabel=show_xlabel, show_yticks=show_yticks)


def _plot_embedding_pair_row(
    ax: Axes,
    df_typ: Optional[pd.DataFrame],
    df_emb: Optional[pd.DataFrame],
    show_xlabel: bool,
    show_yticks: bool,
) -> None:
    """Combined panel: rho(human, emb-metric) and rho(static-emb, emb-metric), one legend."""
    ax.axhline(0.0, color='black', linewidth=0.8, alpha=0.5)
    ax.set_ylim(-1.0, 1.0)
    ax.tick_params(labelsize=TICK_FONTSIZE - 6)
    x = np.arange(len(ordered_cats))

    has_any = False
    for df, color, label in ((df_typ, EMB_TYP_COLOR, 'Human'), (df_emb, EMB_EMB_COLOR, 'Static emb')):
        if df is None or 'spearman_rho' not in df.columns:
            continue
        y = df.set_index('category').reindex(ordered_cats)['spearman_rho'].to_numpy(dtype=float)
        ax.plot(x, y, marker='o', linewidth=1.4, markersize=3.0, color=color, label=label)
        has_any = True

    if not has_any:
        draw_blank(ax, 'no data')
        return

    ax.set_xlim(-0.5, len(ordered_cats) - 0.5)
    ax.set_xticks(x)
    ax.set_xticklabels(ordered_cats if show_xlabel else [''] * len(ordered_cats),
                        rotation=45, ha='right', rotation_mode='anchor',
                        fontsize=CATEGORY_TICK_FONTSIZE - 6)
    ax.set_yticklabels([f'{v:.2f}' for v in ax.get_yticks()] if show_yticks else [],
                        fontsize=TICK_FONTSIZE - 6)
    ax.grid(axis='y', alpha=0.25)
    ax.legend(fontsize=LEGEND_FONTSIZE - 4, loc='upper left', frameon=False,
              handlelength=1.0, borderpad=0.2, labelspacing=0.2)
    if show_xlabel:
        ax.set_xlabel('Category', fontsize=LABEL_FONTSIZE - 6)

## Per-(model, SAE label) 7×4 grid figures

One figure per (model, SAE release) found in W&B. Columns = the 4 new metrics
(L1, KL, SAE cosine, SAE dot). Rows, top to bottom:

0. **AMI vs layer** — k-medoids clustering quality (Adjusted Mutual
   Information against the true dataset categories) per layer, for two
   different word representations: **Dense residual** (black, thick,
   no marker; cosine similarity of the prompted residual-stream vector —
   same curve in every column, it doesn't depend on the SAE metric) vs.
   **`{metric} (SAE)`** (colored, with marker; this column's metric applied
   to the SAE latent vectors). Same x-axis (layer) and y-axis (AMI) for
   both, so the two curves are directly comparable — this shows whether the
   SAE-latent metric separates categories better or worse than the plain
   dense embedding at each layer.
1–3. typicality, static-embedding, and contextual-cosine vs metric (by layer)
4. metric (on the continuous SAE activation vector) vs Jaccard (on the same,
   binarized SAE activation vector) — both built over the target word (by layer)
5. layer×layer consistency
6. combined embedding-side panel (static, uncontextualized SAE) showing both
   human-typicality and static-embedding-cosine comparisons against
   categories, distinguished by a legend.

Columns are the 4 metrics: L1, KL, SAE cosine, SAE dot product. "KL" here is
specifically the L1-normalized variant; `main.py` also computes and logs a
softmax-normalized variant (`klsm`) for comparison, but it isn't plotted here.

The y-scale is only labeled on the leftmost column for line-plot rows (AMI,
embedding-side); heatmap rows already only label categories on the leftmost
column. Layer tick numbers are shown on every layerwise row; the "Layer"
axis-caption text is only shown once, on the bottom-most layerwise row.

In [ ]:
for (model, label), tables in sorted(
    model_data.items(),
    key=lambda kv: (MODEL_ORDER.index(kv[0][0]) if kv[0][0] in MODEL_ORDER else 999, kv[0][1]),
):
    pretty = PRETTY_MODEL.get(model, model)
    n_cats = len(ordered_cats)
    heat_size = max(6.0, 0.12 * n_cats)
    fig_width = N_COLS * heat_size + 2.4
    row_heights = [3.0, heat_size, heat_size, heat_size, heat_size, heat_size, 3.6]
    fig_height = sum(row_heights)

    fig, axes = plt.subplots(
        N_ROWS, N_COLS, figsize=(fig_width, fig_height), squeeze=False,
        gridspec_kw={'height_ratios': row_heights},
    )
    fig.suptitle(f'{pretty} — {label}', fontsize=SUPTITLE_FONTSIZE, y=0.995)

    df_ami = tables.get('df_clustering_ami')

    for col_idx, m in enumerate(METRICS):
        keys = per_metric_table_keys(m)
        show_yticks = (col_idx == 0)

        axes[0, col_idx].set_title(METRIC_LABELS[m], fontsize=TITLE_FONTSIZE - 2, pad=8)
        _plot_ami_row(axes[0, col_idx], df_ami, m, show_xlabel=True, show_yticks=show_yticks)

        _plot_heatmap_row(axes[1, col_idx], tables.get(keys['typ_by_layer']), show_xlabel=False, show_yticks=show_yticks)
        _plot_heatmap_row(axes[2, col_idx], tables.get(keys['emb_by_layer']), show_xlabel=False, show_yticks=show_yticks)
        _plot_heatmap_row(axes[3, col_idx], tables.get(keys['ctx_by_layer']), show_xlabel=False, show_yticks=show_yticks)
        _plot_heatmap_row(axes[4, col_idx], tables.get(keys['jac_by_layer']), show_xlabel=True, show_yticks=show_yticks)

        _plot_ll_row(axes[5, col_idx], tables.get(keys['ll']), show_xlabel=True, show_yticks=show_yticks)

        _plot_embedding_pair_row(
            axes[6, col_idx], tables.get(keys['typ_emb_agg']), tables.get(keys['emb_emb_agg']),
            show_xlabel=True, show_yticks=show_yticks,
        )

    # One uniform offset for every row's side-text, positioned far enough
    # left to clear rows 1-4's long category names (e.g. "natural earth
    # formation") without sitting needlessly far from the other rows' short
    # numeric y-ticks.
    ROW_LABEL_XOFFSET = -0.85
    for row_idx, title in enumerate(ROW_TITLES):
        axes[row_idx, 0].annotate(
            title, xy=(0, 0.5), xycoords='axes fraction',
            xytext=(ROW_LABEL_XOFFSET, 0.5), textcoords='axes fraction',
            ha='center', va='center', rotation=90,
            fontsize=ROWLABEL_FONTSIZE - 6, fontweight='bold',
            annotation_clip=False,
        )

    fig.subplots_adjust(left=0.10, right=0.94, bottom=0.04, top=0.94, hspace=0.35, wspace=0.10)
    add_shared_colorbar(fig, bottom=0.04, top=0.94)

    safe_model = model.replace('/', '__')
    safe_label = label.replace('/', '__').replace(' ', '_')
    out = OUTPUT_DIR / f'{safe_model}__{safe_label}__additional_metrics.png'
    fig.savefig(out, dpi=75, bbox_inches='tight')  # ~2200 px wide = ~300 dpi at full text width
    plt.show()
    plt.close(fig)
    print(f'Saved: {out}')

## Spearman correlation summary tables

One table per (model, SAE label). Columns are the metrics (Jaccard, L1, KL,
SAE cosine, SAE dot); rows are the six comparisons from the grid above,
each collapsed to a single number by averaging the Spearman rho over every
layer and category (the two "emb" rows have no layer dimension, so only
categories are averaged, matching the static/uncontextualized SAE pass).
"ρ(metric, Jaccard)" is undefined for the Jaccard column itself -- shown as "-".

In [ ]:
def _stats_by_layer(df: Optional[pd.DataFrame]) -> Tuple[Optional[float], Optional[float]]:
    """Flatten all (category, layer) Spearman values in a by-layer table and
    return (mean, std) across all of them."""
    if df is None:
        return None, None
    lc = get_layer_cols(df)
    if not lc:
        return None, None
    vals = df[lc].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None, None
    return float(vals.mean()), float(vals.std())


def _stats_over_categories(df: Optional[pd.DataFrame]) -> Tuple[Optional[float], Optional[float]]:
    """(mean, std) of a single-layer (category-only) aggregate table's spearman_rho column."""
    if df is None or 'spearman_rho' not in df.columns:
        return None, None
    vals = df['spearman_rho'].to_numpy(dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None, None
    return float(vals.mean()), float(vals.std())


def _fmt(v) -> str:
    if v is None:
        return '—'
    if isinstance(v, str):
        return v
    if isinstance(v, float) and not np.isfinite(v):
        return '—'
    return f'{v:.3f}'


def _fmt_meanstd(mean: Optional[float], std: Optional[float]) -> str:
    if mean is None:
        return '—'
    return f'{mean:.3f}±{std:.3f}'


SPEARMAN_TABLE_COLUMNS = ['jaccard'] + METRICS  # jaccard, l1, kl, saecos, dot
SPEARMAN_COLUMN_LABELS = {'jaccard': 'Jaccard', **METRIC_LABELS}
SPEARMAN_ROW_LABELS = [
    'ρ(human, metric)',
    'ρ(static-emb, metric)',
    'ρ(ctx-cosine, metric)',
    'ρ(metric, Jaccard)',
    'ρ(human, emb-metric)',
    'ρ(static-emb, emb-metric)',
]


def build_spearman_summary_table(tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Each cell is 'mean±std' of the Spearman rho, flattened across every
    layer and category (categories only for the two 'emb' rows)."""
    out = {}
    for col in SPEARMAN_TABLE_COLUMNS:
        if col == 'jaccard':
            keys = JACCARD_TABLE_KEYS
            jac_stats = (None, None)  # "metric vs Jaccard" is undefined for Jaccard itself
        else:
            keys = per_metric_table_keys(col)
            jac_stats = _stats_by_layer(tables.get(keys['jac_by_layer']))

        stats = [
            _stats_by_layer(tables.get(keys['typ_by_layer'])),
            _stats_by_layer(tables.get(keys['emb_by_layer'])),
            _stats_by_layer(tables.get(keys['ctx_by_layer'])),
            jac_stats,
            _stats_over_categories(tables.get(keys['typ_emb_agg'])),
            _stats_over_categories(tables.get(keys['emb_emb_agg'])),
        ]
        cells = [
            '-' if (col == 'jaccard' and i == 3) else _fmt_meanstd(mean, std)
            for i, (mean, std) in enumerate(stats)
        ]
        out[SPEARMAN_COLUMN_LABELS[col]] = cells

    return pd.DataFrame(out, index=SPEARMAN_ROW_LABELS)


for (model, label), tables in sorted(
    model_data.items(),
    key=lambda kv: (MODEL_ORDER.index(kv[0][0]) if kv[0][0] in MODEL_ORDER else 999, kv[0][1]),
):
    pretty = PRETTY_MODEL.get(model, model)
    print(f'\n=== {pretty} — {label} ===')
    print(build_spearman_summary_table(tables).to_string())

## AMI summary table (across all models/SAE types)

Columns are the (model, SAE label) combinations found in W&B; rows are the
six representations used for k-medoids clustering AMI (dense residual --
the "Dense residual" black curve from the AMI row of the grid above --
Jaccard, L1, KL, SAE cosine, SAE dot), each contributing two rows: the max
AMI across layers (with the layer it occurred at, "ℓ=..."), and the
mean ± std AMI across layers.

In [ ]:
# "contextual" = the dense (prompted) residual-stream cosine-similarity
# clustering -- the same "Dense residual" black curve plotted in the AMI row
# of the grid above. It has a genuine per-layer AMI, unlike the separate
# "static_embedding" representation (uncontextualized W_E, layer-independent,
# not shown in the plots) which is deliberately not included here.
AMI_METRICS = ['contextual', 'jaccard', 'l1', 'kl', 'saecos', 'dot']
AMI_METRIC_LABELS = {
    'contextual': 'Dense residual', 'jaccard': 'Jaccard',
    'l1': 'L1', 'kl': 'KL', 'saecos': 'SAE cos', 'dot': 'SAE dot',
}


def _ami_stats(
    df_ami: Optional[pd.DataFrame], metric: str,
) -> Tuple[Optional[float], Optional[int], Optional[float], Optional[float]]:
    """(max, layer-of-max, mean, std) AMI across layers for this metric's
    clustering representation."""
    if df_ami is None or 'representation' not in df_ami.columns:
        return None, None, None, None
    pattern = rf'^{re.escape(metric)}_layer_\d+$'
    rows = df_ami[df_ami['representation'].str.match(pattern)]
    if len(rows) == 0:
        return None, None, None, None
    layers_ = rows['representation'].str.extract(r'_layer_(\d+)$')[0].astype(int).to_numpy()
    vals = rows['ami'].to_numpy(dtype=float)
    argmax_idx = int(np.argmax(vals))
    return float(vals[argmax_idx]), int(layers_[argmax_idx]), float(vals.mean()), float(vals.std())


def _fmt_max(v: Optional[float], layer: Optional[int]) -> str:
    if v is None:
        return '—'
    if layer is None:
        return f'{v:.3f}'
    return f'{v:.3f} (ℓ={layer})'


_col_keys = sorted(
    model_data.keys(),
    key=lambda k: (MODEL_ORDER.index(k[0]) if k[0] in MODEL_ORDER else 999, k[1]),
)
_col_names = [f'{PRETTY_MODEL.get(m, m)} / {l}' for m, l in _col_keys]

_rows_out: List[List[str]] = []
_row_labels: List[str] = []
for metric in AMI_METRICS:
    max_row, meanstd_row = [], []
    for (model, label) in _col_keys:
        df_ami = model_data[(model, label)].get('df_clustering_ami')
        mx, mx_layer, mn, sd = _ami_stats(df_ami, metric)
        max_row.append(_fmt_max(mx, mx_layer))
        meanstd_row.append(_fmt_meanstd(mn, sd))
    _rows_out.append(max_row);     _row_labels.append(f'{AMI_METRIC_LABELS[metric]} (max)')
    _rows_out.append(meanstd_row); _row_labels.append(f'{AMI_METRIC_LABELS[metric]} (mean±std)')

df_ami_summary = pd.DataFrame(_rows_out, columns=_col_names, index=_row_labels)
print(df_ami_summary.to_string())